# 16 — Confronto Finale per la Tesi: FCGR Baseline vs FCGR Ottimizzate

## Scopo del Notebook
Raccogliere e confrontare tutte le metriche sperimentali ottenute tra la **FCGR Baseline ($k=6$, Notebook `02` e `04`)** e le **Varianti FCGR Ottimizzate (Notebook `05` e `06`)**:
1. **Efficienza Computazionale (Fase 3)**: Tempo di conversione (`ms/seq`), Speedup (`x`), Memory Footprint (`KB`/`MB`/`GB`) e abbattimento della Sparsità Spaziale ($S\%$).
2. **Validazione Downstream Binaria (Hotspot vs Stabile) sulla Rete Siamese (`SiameseV3`, Fase 4)**:
   - **ROC-AUC** e **PR-AUC** (sia Pairwise che Classificazione Binaria con Reference Bank);
   - **F1-Score, Macro-F1, Precision, Recall, Balanced Accuracy**;
   - **Separabilità degli Embedding ($d^+$, $d^-$, Gap $\Delta d$, $d'$)**.

In [ ]:
# Su Google Colab esegui il mount del Drive:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('MANIFEST_PATH:', MANIFEST_PATH, '-> exists:', MANIFEST_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())
from src.fcgr import (
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_vectorized,
    normalize_fcgr_log,
    quantize_fcgr_uint8,
    dequantize_fcgr_uint8,
    compute_multiscale_fcgr,
    plot_fcgr_comparison,
)

## 1. Tabella Comparativa di Efficienza Computazionale (Baseline vs Varianti Ottimizzate)
Carichiamo i benchmark generati dai Notebook `02` e `05`.

In [ ]:
k_bench_path = RESULTS_DIR / 'optimized_fcgr_k_benchmark.csv'
var_summary_path = RESULTS_DIR / 'optimized_variants_k6_summary.csv'

if k_bench_path.exists():
    print('=== BENCHMARK SCALABILITA\' PER k (4..10) ===')
    display(pd.read_csv(k_bench_path))

if var_summary_path.exists():
    print('\n=== RIEPILOGO EFFICIENZA VARIANTI A k=6 (60.000 SEQUENZE) ===')
    var_df = pd.read_csv(var_summary_path)
    display(var_df)
else:
    print('Esegui prima il Notebook 05 per generare optimized_variants_k6_summary.csv!')

## 2. Tabella e Grafici Comparativi Downstream sulla Rete Siamese (`SiameseV3`)
Confrontiamo i risultati sul Test Set tra:
- `siamese_baseline_k6` (Notebook `04`: FCGR Baseline $k=6$, 1 canale `float32`, `915.5 MB`)
- `siamese_opt_rc_log_k6` (Notebook `06`: Bitwise + Reverse-Complement + Log1p, 1 canale `uint8`, `228.9 MB`)
- `siamese_opt_multiscale_k456` (Notebook `06`: Stack Frattale Multi-Scala $k=4,5,6$ + RC + Log1p, 3 canali `uint8`, `686.6 MB`)

In [ ]:
metric_files = (
    sorted(ARTIFACTS_DIR.glob('siamese_baseline_*/*_final_metrics.csv'))
    + sorted(ARTIFACTS_DIR.glob('siamese_opt*/*_final_metrics.csv'))
)

if len(metric_files) > 0:
    dfs = []
    for f in metric_files:
        df_m = pd.read_csv(f)
        df_m.insert(0, 'experiment', f.parent.name)
        dfs.append(df_m)
    comp_df = pd.concat(dfs, ignore_index=True)
    comp_df.to_csv(COMPARISON_DIR / 'siamese_binary_comparison_table.csv', index=False)
    print('=== TABELLA COMPARATIVA FINALE DOWNSTREAM (TEST SET) ===')
    cols_focus = [
        'experiment', 'cls_roc_auc', 'cls_pr_auc', 'cls_f1', 'cls_balanced_accuracy',
        'pairwise_roc_auc', 'd_pos_mean', 'd_neg_mean', 'distance_gap', 'd_prime', 'best_epoch'
    ]
    cols_present = [c for c in cols_focus if c in comp_df.columns]
    display(comp_df[cols_present])

    if len(comp_df) >= 2:
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
        x = np.arange(len(comp_df))
        names = comp_df['experiment'].str.replace('siamese_', '', regex=False).tolist()

        # 1. ROC-AUC & Balanced Accuracy
        w = 0.28
        axes[0].bar(x - w, comp_df['cls_roc_auc'], width=w, label='Classif. ROC-AUC', color='#2B6CB0')
        axes[0].bar(x, comp_df['cls_balanced_accuracy'], width=w, label='Balanced Accuracy', color='#2F855A')
        axes[0].bar(x + w, comp_df['pairwise_roc_auc'], width=w, label='Pairwise ROC-AUC', color='#DD6B20')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(names, rotation=12)
        axes[0].set_ylim(0.5, 1.0)
        axes[0].set_title('Performance Predittiva (Test Set)')
        axes[0].legend()
        axes[0].grid(axis='y', alpha=0.3)

        # 2. Separation Gap & d-prime
        axes[1].bar(x - w/2, comp_df['distance_gap'], width=w, label='Distance Gap (d- - d+)', color='#805AD5')
        axes[1].bar(x + w/2, comp_df['d_prime'], width=w, label="Discriminability (d')", color='#D53F8C')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(names, rotation=12)
        axes[1].set_title('Separabilità dello Spazio di Embedding')
        axes[1].legend()
        axes[1].grid(axis='y', alpha=0.3)

        # 3. F1 & PR-AUC
        axes[2].bar(x - w/2, comp_df['cls_f1'], width=w, label='Hotspot F1-Score', color='#319795')
        axes[2].bar(x + w/2, comp_df['cls_pr_auc'], width=w, label='Classif. PR-AUC', color='#3182CE')
        axes[2].set_xticks(x)
        axes[2].set_xticklabels(names, rotation=12)
        axes[2].set_ylim(0.5, 1.0)
        axes[2].set_title('Precision-Recall & F1 (Hotspot vs Stabile)')
        axes[2].legend()
        axes[2].grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig(COMPARISON_DIR / 'final_thesis_siamese_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print('Nessun file di metriche trovato in artifacts/. Esegui i Notebook 04 e 06!')